# Clase — Visualización interactiva y dashboards: Plotly, Dash y Streamlit

**Bases de Datos Masivas (11088) — 2026**

Hasta ahora los gráficos los hicimos con `matplotlib` y `seaborn`: son **imágenes**. Sirven
para un informe, pero no se puede pasar el mouse por un punto, hacer zoom ni filtrar. En esta
clase vemos tres herramientas que resuelven eso, cada una en un escalón distinto:

| Herramienta | Qué es | Dónde corre | Idea central |
| --- | --- | --- | --- |
| **Plotly** | Librería de gráficos interactivos | En el notebook o en cualquier HTML | Una figura es un **diccionario** (JSON) que dibuja `plotly.js` en el navegador |
| **Dash** | Framework para armar aplicaciones web con Python | Servidor web (Flask) + navegador (React) | Los componentes se conectan con **callbacks**: "cuando cambia esto, recalculá aquello" |
| **Streamlit** | Framework para armar aplicaciones web con Python | Servidor propio + navegador | El script **se vuelve a ejecutar entero, de arriba abajo**, con cada interacción |

Dash y Streamlit usan Plotly para los gráficos, así que el orden de la clase es ese:
primero aprendemos a hacer buenas figuras y después las metemos en un dashboard.

| # | Tema | Dataset |
| --- | --- | --- |
| 1 | Plotly Express y `graph_objects` | alquileres temporarios CABA, clima SMN, gapminder |
| 2 | Dash: layout, callbacks, crossfilter, app completa | alquileres temporarios CABA |
| 3 | Streamlit: modelo de ejecución, widgets, caché, estado | alquileres temporarios CABA |
| 4 | ¿Dash o Streamlit? | — |

Los datasets son los del TP01 (`TPs/TP01/data/`), así que ya los conocen.

## 0. Librerías y configuración

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

SEMILLA = 2026
DATA = Path("../../TPs/TP01/data")   # rutas relativas a jupyterlab/notebooks/

pd.set_option("display.max_columns", 30)
pd.set_option("display.precision", 2)

# Plantilla visual para todas las figuras de la clase
px.defaults.template = "plotly_white"
px.defaults.height = 450

import dash, plotly, streamlit
print("plotly", plotly.__version__, "| dash", dash.__version__, "| streamlit", streamlit.__version__)

plotly 6.9.0 | dash 4.4.1 | streamlit 1.62.0


### Los datos

**Alquileres temporarios en CABA** (`alquileres.csv`): una fila por publicación, con barrio,
tipo de alojamiento, capacidad, precio por noche en pesos, reseñas y coordenadas.

In [2]:
alq_raw = pd.read_csv(DATA / "alquileres.csv")
print("Dimensiones:", alq_raw.shape)
alq_raw.head()

Dimensiones: (29438, 16)


,id,barrio,tipo_propiedad,tipo_habitacion,capacidad,banios,dormitorios,camas,precio_ars,minimo_noches,cantidad_resenias,puntaje_resenias,superanfitrion,latitud,longitud,noches_ocupadas_anio
0,11508,Palermo,Entire condo,Entire home/apt,2,1.0,1.0,1.0,185388.0,3.0,52,4.83,si,-34.58,-58.42,36
1,13851,Belgrano,Entire condo,Entire home/apt,2,1.0,1.0,1.0,101945.0,2.0,1,5.00,no,-34.56,-58.44,0
2,14222,Palermo,Entire rental unit,Entire home/apt,2,NaN,1.0,NaN,NaN,6.0,131,4.78,no,-34.59,-58.41,36
3,20062,Palermo,Entire rental unit,Entire home/apt,4,1.5,2.0,2.0,235938.0,2.0,388,4.94,si,-34.58,-58.44,255
4,24713,San Nicolas,Entire rental unit,Entire home/apt,3,1.0,1.0,2.0,113841.0,1.0,393,4.67,si,-34.60,-58.38,54


Como vimos en el TP01, el precio tiene valores faltantes y una cola larguísima a la derecha.
Para que los gráficos se lean, sacamos las filas sin precio y recortamos el 1 % de cada punta.
**Esto es una decisión de visualización, no de limpieza**: en un análisis real habría que
justificarlo.

In [3]:
p01, p99 = alq_raw.precio_ars.quantile([0.01, 0.99])

alq = (alq_raw
       .dropna(subset=["precio_ars"])
       .query("@p01 <= precio_ars <= @p99")
       .reset_index(drop=True))

# Nos quedamos con los 10 barrios con más publicaciones para los gráficos por barrio
TOP_BARRIOS = alq.barrio.value_counts().head(10).index.tolist()

print(f"Filas: {len(alq_raw):,} → {len(alq):,}  |  precio entre ${p01:,.0f} y ${p99:,.0f}")
print("Top 10 barrios:", TOP_BARRIOS)

Filas: 29,438 → 26,969  |  precio entre $22,868 y $668,652
Top 10 barrios: ['Palermo', 'Recoleta', 'San Nicolas', 'Monserrat', 'Retiro', 'Belgrano', 'Balvanera', 'Almagro', 'Villa Crespo', 'San Telmo']


**Clima del SMN** (`clima_smn.csv`): mediciones horarias de 40 estaciones meteorológicas,
dos días por mes durante un año.

In [4]:
clima = pd.read_csv(DATA / "clima_smn.csv", parse_dates=["fecha"])
clima["momento"] = clima.fecha + pd.to_timedelta(clima.hora, unit="h")
print("Dimensiones:", clima.shape, "| estaciones:", clima.estacion_meteorologica.nunique())
clima.head()

Dimensiones: (23019, 10) | estaciones: 40


,fecha,hora,estacion_meteorologica,temporada,temperatura,humedad,presion,dir_viento,vel_viento,momento
0,2025-09-10,0,AEROPARQUE AERO,Primavera,15.6,80.0,1016.7,340.0,9.0,2025-09-10 00:00:00
1,2025-09-10,1,AEROPARQUE AERO,Primavera,15.3,84.0,1016.1,340.0,9.0,2025-09-10 01:00:00
2,2025-09-10,2,AEROPARQUE AERO,Primavera,15.1,85.0,1015.8,340.0,9.0,2025-09-10 02:00:00
3,2025-09-10,3,AEROPARQUE AERO,Primavera,14.8,85.0,1015.2,340.0,9.0,2025-09-10 03:00:00
4,2025-09-10,4,AEROPARQUE AERO,Primavera,15.0,74.0,1015.0,320.0,17.0,2025-09-10 04:00:00


---

## 1. Plotly

Plotly tiene dos interfaces:

- **`plotly.express` (`px`)**: una función por tipo de gráfico, que recibe un `DataFrame` y
  los nombres de las columnas. Es el equivalente a `seaborn`: con una línea sale un gráfico
  completo. La vamos a usar el 90 % del tiempo.
- **`plotly.graph_objects` (`go`)**: la interfaz de bajo nivel, donde se arma la figura traza
  por traza. Es el equivalente a `matplotlib`. Hace falta para gráficos combinados.

### 1.1 Una figura es un diccionario

Antes de hacer gráficos lindos conviene entender qué es una figura de Plotly, porque eso
explica todo lo demás (incluido cómo funcionan Dash y Streamlit).

In [5]:
mini = alq.groupby("tipo_habitacion", as_index=False).size()
fig = px.bar(mini, x="tipo_habitacion", y="size", title="Publicaciones por tipo")
fig

In [6]:
# Lo que se dibujó arriba es SOLAMENTE esto: un diccionario con dos claves
d = fig.to_dict()
print("claves:", list(d.keys()))
print("cantidad de trazas:", len(d["data"]))
print("tipo    →", d["data"][0]["type"])
print("x       →", list(d["data"][0]["x"]))
print("y       →", d["data"][0]["y"])
print("título  →", d["layout"]["title"]["text"])

claves: ['data', 'layout']
cantidad de trazas: 1
tipo    → bar
x       → ['Entire home/apt', 'Hotel room', 'Private room', 'Shared room']
y       → {'dtype': 'i2', 'bdata': 'dV99AAQJYwA='}
título  → Publicaciones por tipo


- **`data`**: una lista de **trazas** (*traces*). Cada traza es una serie de datos con su tipo
  de marca: `bar`, `scatter`, `box`, `histogram`, `scattermap`…
- **`layout`**: todo lo que no son datos: títulos, ejes, leyenda, colores, tamaño.

Las `y` aparecen raras: desde Plotly 6, los arreglos numéricos viajan **codificados en
binario** (`bdata`, en base64, con su tipo en `dtype`) en lugar de como una lista de números.
Es una optimización: con miles de puntos, el JSON pesa mucho menos.

Python arma ese diccionario, lo convierte en JSON y la librería JavaScript `plotly.js` lo
dibuja en el navegador. Por eso los gráficos son interactivos (zoom, *hover*, ocultar series
haciendo clic en la leyenda) **sin que Python intervenga**: toda la interacción ocurre en
el navegador.

Modificar una figura es modificar ese diccionario:

In [7]:
fig.update_traces(marker_color="seagreen")
fig.update_layout(title="Publicaciones por tipo de habitación", xaxis_title=None, yaxis_title="publicaciones")
fig

### 1.2 Dispersión (*scatter*)

Los parámetros de `px` siempre se leen igual: `x`, `y` son columnas; `color`, `size`,
`symbol` y `facet_*` agregan dimensiones; `hover_data` define qué se muestra al pasar el mouse.

Con 27 mil puntos el gráfico se vuelve una mancha y el notebook pesa mucho, así que tomamos
una muestra.

In [8]:
muestra = alq.sample(4000, random_state=SEMILLA)

fig = px.scatter(
    muestra,
    x="capacidad", y="precio_ars",
    color="tipo_habitacion",
    hover_data=["barrio", "dormitorios", "puntaje_resenias"],
    opacity=0.5,
    labels={"precio_ars": "precio por noche ($)", "capacidad": "huéspedes",
            "tipo_habitacion": "tipo"},
    title="Precio según capacidad",
)
fig

Prueben: pasar el mouse por un punto, seleccionar un rectángulo para hacer zoom, doble clic
para volver, clic en un tipo de la leyenda para ocultarlo, doble clic para dejarlo solo.

La variable `capacidad` es discreta y los puntos se apilan en columnas. Para ver la tendencia
podemos agregar una línea de regresión por grupo (`trendline` usa `statsmodels`) y un
*jitter* manual en x:

In [9]:
m = muestra.query("tipo_habitacion in ['Entire home/apt', 'Private room']").copy()
m["capacidad_j"] = m.capacidad + np.random.default_rng(SEMILLA).uniform(-0.3, 0.3, len(m))

px.scatter(m, x="capacidad_j", y="precio_ars", color="tipo_habitacion",
           trendline="ols", opacity=0.35,
           labels={"capacidad_j": "huéspedes", "precio_ars": "precio por noche ($)"},
           title="Precio según capacidad, con recta de regresión por tipo")

### 1.3 Distribuciones: histograma y boxplot

Los mismos gráficos del TP00, ahora interactivos. `marginal` agrega un boxplot arriba del
histograma, y en el boxplot el *hover* muestra los cinco números sin tener que calcularlos.

In [10]:
px.histogram(alq, x="precio_ars", nbins=60, marginal="box",
             labels={"precio_ars": "precio por noche ($)"},
             title="Distribución del precio por noche")

In [11]:
top = alq[alq.barrio.isin(TOP_BARRIOS)]
orden = top.groupby("barrio").precio_ars.median().sort_values(ascending=False).index

px.box(top, x="barrio", y="precio_ars", color="barrio",
       category_orders={"barrio": list(orden)}, points=False,
       labels={"precio_ars": "precio por noche ($)"},
       title="Precio por barrio (top 10 en publicaciones, ordenados por mediana)"
      ).update_layout(showlegend=False)

### 1.4 Agregaciones y facetas

`px` no agrega por su cuenta (salvo `histogram`): el `groupby` lo hacemos con pandas y le
pasamos el resultado. Las **facetas** (`facet_col`, `facet_row`) repiten el gráfico por cada
valor de una variable, con los ejes compartidos para poder comparar.

In [12]:
resumen = (top.groupby(["barrio", "superanfitrion"], as_index=False)
              .agg(precio_mediano=("precio_ars", "median"),
                   publicaciones=("id", "count"),
                   puntaje=("puntaje_resenias", "mean")))
resumen.head()

,barrio,superanfitrion,precio_mediano,publicaciones,puntaje
0,Almagro,no,82416.0,565,4.71
1,Almagro,si,86852.0,419,4.87
2,Balvanera,no,82386.0,730,4.63
3,Balvanera,si,84868.0,327,4.88
4,Belgrano,no,104388.0,744,4.69


In [13]:
px.bar(resumen, x="barrio", y="precio_mediano", color="superanfitrion",
       barmode="group", hover_data=["publicaciones", "puntaje"],
       category_orders={"barrio": list(orden)},
       labels={"precio_mediano": "precio mediano ($)"},
       title="¿Cobran más los superanfitriones?")

In [14]:
px.histogram(top.query("tipo_habitacion != 'Hotel room'"),
             x="puntaje_resenias", facet_col="tipo_habitacion", histnorm="percent",
             nbins=30, range_x=[3, 5],
             labels={"puntaje_resenias": "puntaje"},
             title="Puntaje de reseñas según tipo (en % dentro de cada tipo)")

`histnorm="percent"` es importante acá: hay 20 veces más departamentos enteros que
habitaciones privadas, y sin normalizar no se podrían comparar las formas.

### 1.5 Series de tiempo

Con el clima armamos la temperatura media diaria de algunas estaciones. `px.line` dibuja una
línea por cada valor de `color`, y el *range slider* permite recorrer el eje temporal.

In [15]:
ESTACIONES = ["USHUAIA AERO", "BUENOS AIRES OBSERVATORIO", "SALTA AERO", "MENDOZA AERO"]

diaria = (clima[clima.estacion_meteorologica.isin(ESTACIONES)]
          .groupby(["estacion_meteorologica", "fecha"], as_index=False)
          .agg(temp_media=("temperatura", "mean"),
               temp_min=("temperatura", "min"),
               temp_max=("temperatura", "max")))

fig = px.line(diaria, x="fecha", y="temp_media", color="estacion_meteorologica",
              markers=True, labels={"temp_media": "°C", "estacion_meteorologica": "estación"},
              title="Temperatura media diaria")
fig.update_xaxes(rangeslider_visible=True)
fig

Y el ciclo diario: temperatura promedio por hora del día y por temporada en una estación.

In [16]:
ciclo = (clima.query("estacion_meteorologica == 'BUENOS AIRES OBSERVATORIO'")
              .groupby(["temporada", "hora"], as_index=False).temperatura.mean())

px.line(ciclo, x="hora", y="temperatura", color="temporada",
        category_orders={"temporada": ["Verano", "Otoño", "Invierno", "Primavera"]},
        labels={"temperatura": "°C", "hora": "hora del día"},
        title="Ciclo diario de temperatura — Buenos Aires Observatorio")

### 1.6 Mapas

Como el dataset trae `latitud` y `longitud`, podemos ubicar cada publicación en el mapa.
`px.scatter_map` usa MapLibre con mapas base abiertos (no hace falta registrarse ni token).
Los mosaicos del mapa se descargan de internet, así que **necesita conexión**.

In [17]:
mapa_muestra = alq.sample(5000, random_state=SEMILLA)

px.scatter_map(mapa_muestra, lat="latitud", lon="longitud",
               color="precio_ars", color_continuous_scale="Viridis",
               hover_name="barrio", hover_data=["tipo_habitacion", "capacidad"],
               zoom=11.2, height=600, map_style="carto-positron", opacity=0.6,
               labels={"precio_ars": "$ / noche"},
               title="Publicaciones coloreadas por precio")

Cuando los puntos se superponen, un mapa de **densidad** muestra mejor dónde se concentra la
oferta:

In [18]:
px.density_map(alq.sample(12000, random_state=SEMILLA), lat="latitud", lon="longitud",
               radius=6, zoom=11.2, height=600, map_style="carto-positron",
               title="Densidad de publicaciones")

### 1.7 Animación

`animation_frame` genera un cuadro por cada valor de una columna. El ejemplo clásico es
*gapminder* (viene incluido en Plotly): esperanza de vida contra PBI per cápita, de 1952 a
2007. Denle *play*.

In [19]:
gap = px.data.gapminder()

px.scatter(gap, x="gdpPercap", y="lifeExp", size="pop", color="continent",
           hover_name="country", animation_frame="year", animation_group="country",
           log_x=True, size_max=55, range_x=[100, 100_000], range_y=[25, 90],
           labels={"gdpPercap": "PBI per cápita (USD, log)", "lifeExp": "esperanza de vida"},
           title="Gapminder")

### 1.8 `graph_objects`: armar la figura a mano

Cuando hay que combinar tipos de gráfico, o usar dos ejes Y, `px` no alcanza. Con
`go.Figure` se agregan trazas una por una (`add_trace`) y `make_subplots` arma grillas.

Ejemplo: temperatura y humedad del mismo día en una estación, cada una con su eje.

In [20]:
dia = clima.query("estacion_meteorologica == 'BUENOS AIRES OBSERVATORIO'"
                  " and fecha == '2026-01-15'")

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=dia.hora, y=dia.temperatura, name="temperatura (°C)",
                         mode="lines+markers", line_color="firebrick"),
              secondary_y=False)
fig.add_trace(go.Bar(x=dia.hora, y=dia.humedad, name="humedad (%)",
                     marker_color="steelblue", opacity=0.35),
              secondary_y=True)
fig.update_layout(title="Buenos Aires, 15/01/2026: temperatura y humedad",
                  xaxis_title="hora", hovermode="x unified")
fig.update_yaxes(title_text="°C", secondary_y=False)
fig.update_yaxes(title_text="%", secondary_y=True, range=[0, 100])
fig

`hovermode="x unified"` muestra todas las series en un solo cartel por cada x: muy útil en
series de tiempo.

Una grilla de subgráficos de distinto tipo:

In [21]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["Precio por noche", "Publicaciones por tipo"])

fig.add_trace(go.Histogram(x=alq.precio_ars, nbinsx=50, name="precio"), row=1, col=1)
fig.add_trace(go.Bar(x=mini.tipo_habitacion, y=mini["size"], name="tipo"), row=1, col=2)

fig.update_layout(showlegend=False, height=400, title="Dos gráficos en una figura")
fig

### Exportar

Una figura se puede guardar como **HTML autocontenido**: se abre en cualquier navegador, sin
Python, y sigue siendo interactiva. Es la forma más simple de compartir un gráfico.
(`include_plotlyjs="cdn"` hace que el archivo pese KB en lugar de MB, a cambio de necesitar
internet para abrirlo).

In [22]:
Path("salidas").mkdir(exist_ok=True)
fig.write_html("salidas/dos_graficos.html", include_plotlyjs="cdn")
print("Guardado:", Path("salidas/dos_graficos.html").resolve())

Guardado: /home/mlulic/desarrollo/bdm2026/jupyterlab/notebooks/salidas/dos_graficos.html


Hasta acá, toda la interacción era **visual**: zoom, *hover*, ocultar series. Pero no podemos
pedirle al gráfico "mostrame sólo Palermo" o "sólo los de menos de $100.000", porque eso
requiere volver a filtrar el `DataFrame`, es decir, **volver a ejecutar Python**. Para eso
necesitamos un servidor: Dash o Streamlit.

---

## 2. Dash

Una app de Dash tiene dos partes:

1. **`layout`**: el árbol de componentes que forman la página. Es HTML escrito en Python.
   - `dash.html` tiene un componente por etiqueta HTML: `html.Div`, `html.H1`, `html.P`…
   - `dash.dcc` (*Dash Core Components*) tiene los componentes interactivos: `dcc.Graph`,
     `dcc.Dropdown`, `dcc.Slider`, `dcc.Checklist`…
2. **callbacks**: funciones de Python que se ejecutan cuando cambia la propiedad de un
   componente, y cuyo resultado se escribe en la propiedad de otro.

Por debajo, Dash levanta un servidor **Flask**, y el navegador dibuja los componentes con
**React**. Cuando el usuario toca un control, el navegador le manda al servidor el valor
nuevo, Python ejecuta el callback y devuelve el resultado (por ejemplo, una figura nueva).

### Correr Dash dentro de JupyterLab

`app.run(jupyter_mode="inline")` levanta el servidor en un hilo del kernel y muestra la app
debajo de la celda. Otras opciones: `"tab"` (la abre en otra pestaña del navegador) y
`"external"` (imprime la URL). Cada app usa **su propio puerto** para que no se pisen.

> Las apps sólo funcionan **mientras el kernel está vivo**. Si abren el notebook y ven un
> recuadro vacío, ejecuten de nuevo la celda.

In [23]:
from dash import Dash, html, dcc, callback, Input, Output
import dash_bootstrap_components as dbc

MODO = "inline"   # probar también "tab" o "external"

### 2.1 La primera app: sólo layout

Sin callbacks: un título, un párrafo y un gráfico. El gráfico es interactivo (es Plotly),
pero no hay nada que lo conecte con Python.

In [24]:
app = Dash(__name__)

app.layout = html.Div([
    html.H2("Alquileres temporarios en CABA"),
    html.P(f"{len(alq):,} publicaciones. Este texto y el gráfico son estáticos."),
    dcc.Graph(figure=px.histogram(alq, x="precio_ars", nbins=50, height=350)),
])

app.run(jupyter_mode=MODO, port=8050)

### 2.2 El primer callback

Agregamos un `dcc.Dropdown` para elegir el barrio. El decorador `@callback` declara:

- **`Input("barrio", "value")`**: cuando cambie la propiedad `value` del componente con
  `id="barrio"`, ejecutá esta función y pasale ese valor como argumento.
- **`Output("grafico", "figure")`**: lo que devuelva la función, escribilo en la propiedad
  `figure` del componente con `id="grafico"`.

Los `id` son la forma de conectar el layout con los callbacks. Dash ejecuta cada callback
una vez al arrancar, para dibujar el estado inicial.

In [25]:
app = Dash(__name__)

app.layout = html.Div([
    html.H3("Distribución del precio por barrio"),
    dcc.Dropdown(id="barrio", options=TOP_BARRIOS, value="Palermo", clearable=False,
                 style={"width": "300px"}),
    dcc.Graph(id="grafico"),
])


@callback(Output("grafico", "figure"), Input("barrio", "value"))
def actualizar_histograma(barrio):
    datos = alq[alq.barrio == barrio]
    return px.histogram(datos, x="precio_ars", nbins=40, height=380,
                        title=f"{barrio}: {len(datos):,} publicaciones, "
                              f"mediana ${datos.precio_ars.median():,.0f}")


app.run(jupyter_mode=MODO, port=8051)

Noten que la función `actualizar_histograma` **nunca la llamamos nosotros**: la llama Dash
cada vez que cambia el dropdown. Es **programación reactiva**: declaramos dependencias y el
framework decide cuándo ejecutar.

### 2.3 Varios inputs y varios outputs

Un callback puede depender de varios controles y actualizar varios componentes a la vez.
Basta con pasar listas. Los argumentos de la función llegan **en el mismo orden** que los
`Input`, y la función devuelve una tupla en el mismo orden que los `Output`.

In [26]:
app = Dash(__name__)

app.layout = html.Div([
    html.H3("Filtros combinados"),
    html.Div([
        html.Div([html.Label("Barrios"),
                  dcc.Dropdown(id="f-barrios", options=TOP_BARRIOS, multi=True,
                               value=["Palermo", "Recoleta", "San Telmo"])],
                 style={"flex": 2}),
        html.Div([html.Label("Tipo de habitación"),
                  dcc.Checklist(id="f-tipo", options=sorted(alq.tipo_habitacion.unique()),
                                value=["Entire home/apt", "Private room"], inline=True)],
                 style={"flex": 3, "paddingLeft": "20px"}),
    ], style={"display": "flex"}),
    html.Label("Precio por noche ($)"),
    dcc.RangeSlider(id="f-precio", min=0, max=600_000, step=10_000, value=[0, 300_000],
                    marks={i: f"{i // 1000}k" for i in range(0, 600_001, 100_000)},
                    tooltip={"placement": "bottom"}),
    html.H4(id="kpi", style={"color": "#2a6f97"}),
    dcc.Graph(id="g-scatter"),
], style={"fontFamily": "sans-serif"})


@callback(
    Output("g-scatter", "figure"),
    Output("kpi", "children"),
    Input("f-barrios", "value"),
    Input("f-tipo", "value"),
    Input("f-precio", "value"),
)
def filtrar(barrios, tipos, rango):
    datos = alq[alq.barrio.isin(barrios or [])
                & alq.tipo_habitacion.isin(tipos or [])
                & alq.precio_ars.between(*rango)]
    fig = px.scatter(datos, x="puntaje_resenias", y="precio_ars", color="barrio",
                     opacity=0.5, height=420,
                     labels={"puntaje_resenias": "puntaje", "precio_ars": "$ / noche"})
    if datos.empty:
        return fig, "Ninguna publicación cumple los filtros"
    texto = (f"{len(datos):,} publicaciones · precio mediano ${datos.precio_ars.median():,.0f}"
             f" · puntaje medio {datos.puntaje_resenias.mean():.2f}")
    return fig, texto


app.run(jupyter_mode=MODO, port=8052)

### 2.4 Un gráfico que filtra a otro (*crossfilter*)

Los `dcc.Graph` también son inputs: sus propiedades `clickData`, `hoverData` y
`selectedData` cambian cuando el usuario hace clic, pasa el mouse o selecciona puntos.
Con eso podemos hacer que **un clic en una barra filtre el mapa**, que es lo que hace
cualquier herramienta de BI (Power BI, Tableau, Superset).

Para ver qué trae `clickData`, la app lo muestra crudo debajo del mapa.

In [27]:
por_barrio = (alq.groupby("barrio", as_index=False)
                 .agg(publicaciones=("id", "count"), precio_mediano=("precio_ars", "median"))
                 .sort_values("publicaciones").tail(15))

fig_barras = px.bar(por_barrio, x="publicaciones", y="barrio", orientation="h",
                    color="precio_mediano", color_continuous_scale="Blues", height=500,
                    title="Hacé clic en un barrio")

app = Dash(__name__)

app.layout = html.Div([
    html.Div([dcc.Graph(id="barras", figure=fig_barras)], style={"flex": 2}),
    html.Div([dcc.Graph(id="mapa"),
              html.Pre(id="click-crudo", style={"fontSize": "11px"})], style={"flex": 3}),
], style={"display": "flex"})


@callback(Output("mapa", "figure"), Output("click-crudo", "children"),
          Input("barras", "clickData"))
def filtrar_mapa(click):
    if click is None:                      # todavía no hubo clic: mostramos todo
        barrio, datos = "todos los barrios", alq.sample(3000, random_state=SEMILLA)
    else:
        barrio = click["points"][0]["y"]   # barras horizontales: el barrio está en y
        datos = alq[alq.barrio == barrio]
    fig = px.scatter_map(datos, lat="latitud", lon="longitud", color="tipo_habitacion",
                         hover_data=["precio_ars"], zoom=11.5 if click else 10.8,
                         map_style="carto-positron", height=460, title=barrio)
    fig.update_layout(legend=dict(orientation="h", y=-0.05))
    return fig, f"clickData = {click}"


app.run(jupyter_mode=MODO, port=8053)

### 2.5 La app completa, con Bootstrap

El layout con `html.Div` y estilos a mano se vuelve tedioso. `dash-bootstrap-components`
(`dbc`) trae componentes de **Bootstrap**: `dbc.Container`, una grilla de 12 columnas con
`dbc.Row` / `dbc.Col`, tarjetas (`dbc.Card`), pestañas, temas visuales ya hechos.

Juntamos todo lo anterior en un dashboard y lo escribimos como **un archivo `.py`** con
`%%writefile`, porque así es como vive una app de Dash en la realidad: un script que se
ejecuta en un servidor, no un notebook. Las rutas se arman a partir de `__file__` para que
funcione desde cualquier carpeta.

In [28]:
Path("apps").mkdir(exist_ok=True)

In [29]:
%%writefile apps/dash_alquileres.py
"""Dashboard de alquileres temporarios en CABA con Dash + Bootstrap.

Correr desde la raíz del repo (bdm2026/):
    uv run --project jupyterlab python jupyterlab/notebooks/apps/dash_alquileres.py
y abrir http://localhost:8060
"""
from pathlib import Path

import dash_bootstrap_components as dbc
import pandas as pd
import plotly.express as px
from dash import Dash, Input, Output, callback, dcc, html

RAIZ = Path(__file__).resolve().parents[3]          # .../bdm2026
DATA = RAIZ / "TPs" / "TP01" / "data"

# ---------------------------------------------------------------- datos
alq = pd.read_csv(DATA / "alquileres.csv").dropna(subset=["precio_ars"])
p01, p99 = alq.precio_ars.quantile([0.01, 0.99])
alq = alq[alq.precio_ars.between(p01, p99)]
BARRIOS = alq.barrio.value_counts().head(15).index.tolist()
TIPOS = sorted(alq.tipo_habitacion.unique())


def tarjeta_kpi(titulo, id_valor):
    return dbc.Card(dbc.CardBody([html.Small(titulo, className="text-muted"),
                                  html.H3(id=id_valor, className="mb-0")]))


# ---------------------------------------------------------------- layout
controles = dbc.Card(dbc.CardBody([
    html.Label("Barrios"),
    dcc.Dropdown(id="barrios", options=BARRIOS, value=BARRIOS[:5], multi=True),
    html.Label("Tipo de habitación", className="mt-3"),
    dcc.Checklist(id="tipos", options=TIPOS, value=TIPOS,
                  inputStyle={"marginRight": "6px"}, labelStyle={"display": "block"}),
    html.Label("Precio por noche ($)", className="mt-3"),
    dcc.RangeSlider(id="precio", min=0, max=int(p99), step=10_000, value=[0, int(p99)],
                    marks=None, tooltip={"placement": "bottom", "always_visible": True}),
    dbc.Switch(id="super", label="Sólo superanfitriones", value=False, className="mt-3"),
]))

app = Dash(__name__, external_stylesheets=[dbc.themes.FLATLY])
app.title = "Alquileres CABA"

app.layout = dbc.Container([
    html.H2("Alquileres temporarios en CABA", className="my-3"),
    dbc.Row([
        dbc.Col(controles, md=3),
        dbc.Col([
            dbc.Row([
                dbc.Col(tarjeta_kpi("Publicaciones", "kpi-n")),
                dbc.Col(tarjeta_kpi("Precio mediano", "kpi-precio")),
                dbc.Col(tarjeta_kpi("Puntaje medio", "kpi-puntaje")),
            ], className="mb-3"),
            dbc.Tabs([
                dbc.Tab(dcc.Graph(id="g-box"), label="Precios"),
                dbc.Tab(dcc.Graph(id="g-mapa"), label="Mapa"),
                dbc.Tab(dcc.Graph(id="g-scatter"), label="Precio vs. puntaje"),
            ]),
        ], md=9),
    ]),
], fluid=True)


# ---------------------------------------------------------------- callbacks
@callback(
    Output("kpi-n", "children"), Output("kpi-precio", "children"),
    Output("kpi-puntaje", "children"),
    Output("g-box", "figure"), Output("g-mapa", "figure"), Output("g-scatter", "figure"),
    Input("barrios", "value"), Input("tipos", "value"),
    Input("precio", "value"), Input("super", "value"),
)
def actualizar(barrios, tipos, rango, solo_super):
    d = alq[alq.barrio.isin(barrios or []) & alq.tipo_habitacion.isin(tipos or [])
            & alq.precio_ars.between(*rango)]
    if solo_super:
        d = d[d.superanfitrion == "si"]

    box = px.box(d, x="barrio", y="precio_ars", color="tipo_habitacion", points=False,
                 labels={"precio_ars": "$ / noche"}, template="plotly_white")
    mapa = px.scatter_map(d.sample(min(len(d), 4000), random_state=2026),
                          lat="latitud", lon="longitud", color="precio_ars",
                          color_continuous_scale="Viridis", zoom=11.3, height=550,
                          map_style="carto-positron", hover_name="barrio")
    scatter = px.scatter(d, x="puntaje_resenias", y="precio_ars", color="barrio",
                         opacity=0.4, template="plotly_white",
                         labels={"precio_ars": "$ / noche", "puntaje_resenias": "puntaje"})

    if d.empty:
        return "0", "—", "—", box, mapa, scatter
    return (f"{len(d):,}", f"${d.precio_ars.median():,.0f}",
            f"{d.puntaje_resenias.mean():.2f}", box, mapa, scatter)


if __name__ == "__main__":
    app.run(debug=True, port=8060)

Overwriting apps/dash_alquileres.py


Desde la terminal, parados en la raíz del repo (`bdm2026/`):

```bash
uv run --project jupyterlab python jupyterlab/notebooks/apps/dash_alquileres.py
```

y abrir <http://localhost:8060>. Con `debug=True` la app se recarga sola cada vez que se
guarda el archivo, y aparece un panel de errores abajo a la derecha.

También la podemos importar y correr acá mismo:

In [30]:
import importlib, sys
sys.path.insert(0, "apps")
import dash_alquileres
importlib.reload(dash_alquileres)       # por si editaron el archivo

dash_alquileres.app.run(jupyter_mode=MODO, port=8060)

**Para recordar de Dash**

- Todo es explícito: cada interacción es un callback con sus `Input` y `Output` declarados.
- Sólo se recalcula lo que depende del control que cambió: con muchos componentes, es
  eficiente.
- El estado vive **en el navegador** (en las propiedades de los componentes), así que el
  servidor no guarda nada por usuario y escala bien con muchos usuarios a la vez.
- A cambio, hay más código para escribir, y el layout lo armamos nosotros.

---

## 3. Streamlit

### 3.1 Un modelo de ejecución distinto

Streamlit no tiene callbacks ni layout separado. Una app es **un script de Python común**,
que se lee de arriba abajo:

```python
import streamlit as st

barrio = st.selectbox("Barrio", ["Palermo", "Recoleta"])   # dibuja un control y devuelve su valor
datos = alq[alq.barrio == barrio]
st.plotly_chart(px.histogram(datos, x="precio_ars"))         # dibuja el gráfico
```

Cada `st.algo(...)` **dibuja** un elemento en la página en el orden en que se ejecuta. Los
widgets, además, **devuelven el valor actual**: `barrio` es directamente un `str`.

¿Y cuando el usuario cambia el selector? **Streamlit vuelve a ejecutar el script entero,
desde la primera línea**, ahora con `st.selectbox` devolviendo el valor nuevo.

| | Dash | Streamlit |
| --- | --- | --- |
| Qué se ejecuta al tocar un control | Sólo el callback que depende de ese control | **Todo el script** |
| Cómo se conectan control y gráfico | Con `id` + `Input`/`Output` | Por el orden del código: es una variable |
| Dónde se define la página | En `app.layout`, aparte de la lógica | Mezclado con la lógica, en el orden del script |

Esto tiene dos consecuencias que hay que manejar:

1. **Si el script lee un CSV, lo lee de nuevo en cada clic.** Para evitarlo está
   `@st.cache_data`: la función se ejecuta una vez y las siguientes devuelve el resultado
   guardado (mientras no cambien sus argumentos).
2. **Las variables no sobreviven entre ejecuciones.** Un `contador = 0` vuelve a 0 en cada
   clic. Para guardar algo entre ejecuciones está `st.session_state`, un diccionario que
   persiste mientras dure la sesión del usuario.

### 3.2 La app

Streamlit no corre dentro del kernel de Jupyter: necesita su propio proceso. Escribimos el
script con `%%writefile` y lo lanzamos aparte. Es el mismo dashboard que el de Dash, para
poder comparar.

In [31]:
%%writefile apps/streamlit_alquileres.py
"""Dashboard de alquileres temporarios en CABA con Streamlit.

Correr desde la raíz del repo (bdm2026/):
    uv run --project jupyterlab streamlit run jupyterlab/notebooks/apps/streamlit_alquileres.py
"""
from pathlib import Path

import pandas as pd
import plotly.express as px
import streamlit as st

RAIZ = Path(__file__).resolve().parents[3]          # .../bdm2026
DATA = RAIZ / "TPs" / "TP01" / "data"

# Tiene que ser la primera llamada a st.* del script
st.set_page_config(page_title="Alquileres CABA", page_icon="🏠", layout="wide")


# ------------------------------------------------------------ datos (con caché)
@st.cache_data
def cargar():
    """Se ejecuta UNA vez; en los reruns siguientes devuelve el DataFrame guardado."""
    df = pd.read_csv(DATA / "alquileres.csv").dropna(subset=["precio_ars"])
    p01, p99 = df.precio_ars.quantile([0.01, 0.99])
    return df[df.precio_ars.between(p01, p99)]


alq = cargar()
BARRIOS = alq.barrio.value_counts().head(15).index.tolist()
TIPOS = sorted(alq.tipo_habitacion.unique())
PRECIO_MAX = int(alq.precio_ars.max())

# ------------------------------------------------------------ barra lateral
st.sidebar.header("Filtros")
barrios = st.sidebar.multiselect("Barrios", BARRIOS, default=BARRIOS[:5])
tipos = st.sidebar.multiselect("Tipo de habitación", TIPOS, default=TIPOS)
rango = st.sidebar.slider("Precio por noche ($)", 0, PRECIO_MAX, (0, PRECIO_MAX), step=10_000)
solo_super = st.sidebar.toggle("Sólo superanfitriones")

# ------------------------------------------------------------ filtrado
# Esto es pandas común: se recalcula en cada interacción, y está bien porque es rápido
d = alq[alq.barrio.isin(barrios) & alq.tipo_habitacion.isin(tipos)
        & alq.precio_ars.between(*rango)]
if solo_super:
    d = d[d.superanfitrion == "si"]

# ------------------------------------------------------------ página
st.title("🏠 Alquileres temporarios en CABA")

if d.empty:
    st.warning("Ninguna publicación cumple los filtros.")
    st.stop()                                         # corta la ejecución del script acá

c1, c2, c3 = st.columns(3)
c1.metric("Publicaciones", f"{len(d):,}")
c2.metric("Precio mediano", f"${d.precio_ars.median():,.0f}",
          delta=f"{d.precio_ars.median() / alq.precio_ars.median() - 1:+.0%} vs. CABA",
          delta_color="inverse")
c3.metric("Puntaje medio", f"{d.puntaje_resenias.mean():.2f}")

tab_precios, tab_mapa, tab_datos = st.tabs(["Precios", "Mapa", "Datos"])

with tab_precios:
    st.plotly_chart(
        px.box(d, x="barrio", y="precio_ars", color="tipo_habitacion", points=False,
               labels={"precio_ars": "$ / noche"}, template="plotly_white"),
        width="stretch")

with tab_mapa:
    st.plotly_chart(
        px.scatter_map(d.sample(min(len(d), 4000), random_state=2026),
                       lat="latitud", lon="longitud", color="precio_ars",
                       color_continuous_scale="Viridis", zoom=11.3, height=600,
                       map_style="carto-positron", hover_name="barrio"),
        width="stretch")

with tab_datos:
    st.dataframe(d, width="stretch", hide_index=True)
    st.download_button("Descargar CSV filtrado", d.to_csv(index=False).encode("utf-8"),
                       file_name="alquileres_filtrados.csv", mime="text/csv")

# ------------------------------------------------------------ el estado entre reruns
with st.expander("¿Por qué hace falta st.session_state?"):
    st.write("Cada botón provoca un rerun completo del script. Compará los dos contadores:")

    contador_comun = 0                                 # se reinicia en cada rerun
    if "contador" not in st.session_state:             # sólo la primera vez de la sesión
        st.session_state.contador = 0

    if st.button("Sumar 1"):
        contador_comun += 1
        st.session_state.contador += 1

    a, b = st.columns(2)
    a.metric("Variable común", contador_comun)
    b.metric("st.session_state", st.session_state.contador)

Overwriting apps/streamlit_alquileres.py


Recorriendo el script de arriba abajo:

| Elemento | Qué hace |
| --- | --- |
| `st.set_page_config` | Título de la pestaña, ícono, layout ancho |
| `@st.cache_data` | La lectura del CSV ocurre una sola vez |
| `st.sidebar.*` | Los widgets van a la barra lateral; cada uno devuelve su valor |
| `st.stop()` | Corta el script: lo que sigue no se dibuja |
| `st.columns` / `st.metric` | Tarjetas de KPI en tres columnas, con variación |
| `st.tabs` + `with` | Lo que se dibuja dentro del `with` va a esa pestaña |
| `st.plotly_chart` | Las mismas figuras de Plotly de la sección 1 |
| `st.dataframe` / `st.download_button` | Tabla navegable y descarga del filtrado |
| `st.session_state` | Estado que sobrevive a los reruns |

### 3.3 Lanzarla desde el notebook

Lo normal es correrla desde una terminal (con la terminal de JupyterLab alcanza):

```bash
uv run --project jupyterlab streamlit run jupyterlab/notebooks/apps/streamlit_alquileres.py
```

Abre el navegador sola en <http://localhost:8501>. Cada vez que se guarda el `.py`, aparece
arriba a la derecha la opción **Rerun / Always rerun** para recargar.

Para la clase, la lanzamos como un proceso aparte desde acá y la embebemos con un `IFrame`:

In [32]:
import subprocess, sys, time, urllib.request
from IPython.display import IFrame

PUERTO_ST = 8501
streamlit_proc = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "apps/streamlit_alquileres.py",
     "--server.headless", "true", "--server.port", str(PUERTO_ST)],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)

# Esperamos a que el servidor responda antes de mostrarlo
for _ in range(60):
    try:
        urllib.request.urlopen(f"http://localhost:{PUERTO_ST}/_stcore/health", timeout=1)
        print(f"Streamlit arriba en http://localhost:{PUERTO_ST}  (pid {streamlit_proc.pid})")
        break
    except OSError:
        time.sleep(0.5)
else:
    print("Streamlit no respondió; revisar corriéndolo desde la terminal")

IFrame(f"http://localhost:{PUERTO_ST}", width="100%", height=750)

Streamlit arriba en http://localhost:8501  (pid 159585)


Prueben el expander del final: el contador común siempre muestra 1 (o 0), el de
`session_state` va acumulando.

Cuando terminen, bajen el proceso (si no, queda ocupando el puerto 8501):

In [33]:
streamlit_proc.terminate()
streamlit_proc.wait(timeout=10)
print("Streamlit detenido, código de salida:", streamlit_proc.returncode)

Streamlit detenido, código de salida: 0


**Para recordar de Streamlit**

- Es el camino más corto de un análisis en pandas a una app: el script se lee como un
  notebook.
- Todo el script se re-ejecuta con cada interacción: lo costoso va en `@st.cache_data`
  (datos) o `@st.cache_resource` (conexiones a bases de datos, modelos).
- El estado vive **en el servidor**, por sesión (`st.session_state`): cada usuario conectado
  ocupa memoria del servidor.
- El layout es más rígido (barra lateral, columnas, pestañas, contenedores): se hace rápido
  algo prolijo, pero cuesta salirse del molde.

---

## 4. ¿Dash o Streamlit?

| Criterio | Dash | Streamlit |
| --- | --- | --- |
| Curva de aprendizaje | Media: hay que entender callbacks e `id` | Baja: es un script |
| Cantidad de código para la misma app | Más (layout + callbacks) | Menos |
| Qué se recalcula | Sólo lo que depende del control | Todo el script (mitigado con caché) |
| Control del diseño | Total (HTML, CSS, Bootstrap) | Limitado a los contenedores que ofrece |
| Interacción entre gráficos (*crossfilter*) | Nativa (`clickData`, `selectedData`) | Posible (`on_select` en `st.plotly_chart`), más limitada |
| Estado | En el navegador | En el servidor, por sesión |
| Muchos usuarios simultáneos | Escala bien (servidor sin estado) | Más consumo por usuario |
| Despliegue | Cualquier servidor WSGI (gunicorn), Docker | `streamlit run`, Docker, Streamlit Community Cloud |
| Caso típico | Dashboard institucional, muchos usuarios, diseño a medida | Prototipo, herramienta interna, demo de un modelo |

Regla práctica: **empezar con Streamlit** para explorar y mostrar rápido; pasar a **Dash**
cuando el dashboard se vuelve un producto (muchos usuarios, interacción fina entre gráficos,
diseño definido por otros).

En los dos casos, el gráfico es Plotly: lo que aprendieron en la sección 1 se usa igual.